# 01 - Load and Clean GRACE Basin-Month Data

This notebook aggregates GRACE/GRACE-FO mascon NetCDF data to Amazon basin polygons. If a processed basin-month CSV already exists, it loads and checks that file instead.

In [1]:
from pathlib import Path
import sys
# Resolve imports when the notebook runs from either the repo root or notebooks folder.
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

from grace_gnn.config import DATA_RAW, BASIN_MONTH_CSV, PODAAC_GRACE_SHORT_NAME, ensure_dirs
from grace_gnn.data import find_first_file, find_mask_zips, read_basin_month_csv, aggregate_grace_netcdf_to_basins, aggregate_grace_netcdf_to_mask_zips, download_grace_from_podaac

ensure_dirs()
print('Raw data folder:', DATA_RAW)
# Prepare expected data folders before any download or aggregation work.
print('Processed basin-month CSV:', BASIN_MONTH_CSV)
DOWNLOAD_GRACE_FROM_PODAAC = True
# The supplied L2 mask zips define coarse basin nodes. Keep None for all L2 basins.
MASK_BASIN_NAME_FILTER = None

Raw data folder: C:\Users\ishaa\Downloads\sees\data\raw
Processed basin-month CSV: C:\Users\ishaa\Downloads\sees\data\processed\basin_month_grace.csv


In [2]:
# Reuse the processed basin-month table when it is already available.
if BASIN_MONTH_CSV.exists():
    basin_month = read_basin_month_csv(BASIN_MONTH_CSV)
    print(f'Loaded existing {BASIN_MONTH_CSV} with {len(basin_month):,} rows')
else:
    grace_nc = find_first_file(DATA_RAW, ['.nc', '.nc4'])
    # Try an authenticated PO.DAAC download only when no local NetCDF is present.
    if grace_nc is None and DOWNLOAD_GRACE_FROM_PODAAC:
        try:
            downloaded = download_grace_from_podaac(DATA_RAW, PODAAC_GRACE_SHORT_NAME)
            grace_nc = next((p for p in downloaded if p.suffix.lower() in ['.nc', '.nc4']), None)
        except Exception as exc:
            print('PO.DAAC download did not complete:', exc)
            grace_nc = None
    basin_file = find_first_file(DATA_RAW, ['.shp', '.geojson', '.gpkg'])
    mask_zips = find_mask_zips(DATA_RAW) + find_mask_zips(ROOT / 'masks')
    # Locate either polygon boundaries or mask zips to define basin-level aggregation.
    if grace_nc is None or (basin_file is None and not mask_zips):
        print('Required raw files were not found.')
        print('Download/place a GRACE mascon NetCDF and either HydroBASINS polygons or .mask.xyz zip files.')
        print('GRACE: https://podaac.jpl.nasa.gov/dataset/TELLUS_GRAC-GRFO_MASCON_GRID_RL06.3_V4')
        print('HydroBASINS: https://www.hydrosheds.org/products/hydrobasins')
        basin_month = None
    else:
        print('Using GRACE file:', grace_nc)
        if basin_file is not None:
            print('Using basin polygon file:', basin_file)
        # Aggregate GRACE mascons through the available basin geometry source.
            basin_month = aggregate_grace_netcdf_to_basins(grace_nc, basin_file, BASIN_MONTH_CSV, amazon_main_bas_id=None)
        else:
            print('Using mask zip files:', mask_zips)
            basin_month = aggregate_grace_netcdf_to_mask_zips(grace_nc, mask_zips, BASIN_MONTH_CSV, basin_name_filter=MASK_BASIN_NAME_FILTER)
        print(f'Saved {len(basin_month):,} rows to {BASIN_MONTH_CSV}')

Loaded existing C:\Users\ishaa\Downloads\sees\data\processed\basin_month_grace.csv with 18,176 rows


In [3]:
if basin_month is not None and not basin_month.empty:
    # Show a quick sanity check of the processed basin-month coverage.
    display(basin_month.head())
    print('Date range:', basin_month['date'].min(), 'to', basin_month['date'].max())
    print('Basins:', basin_month['basin_id'].nunique())

,date,basin_id,basin_name,twsa_cm
0,2002-04-17 12:00:00,10020000010,Antarctic Peninsula Coastal,14.102845
1,2002-05-10 12:00:00,10020000010,Antarctic Peninsula Coastal,15.644680
2,2002-08-16 12:00:00,10020000010,Antarctic Peninsula Coastal,10.975014
3,2002-09-16 00:00:00,10020000010,Antarctic Peninsula Coastal,11.192149
4,2002-10-16 12:00:00,10020000010,Antarctic Peninsula Coastal,13.315436


Date range: 2002-04-17 12:00:00 to 2026-04-16 00:00:00
Basins: 71
